# summarizationmiddleware ミドルウェア
## trigger keep

In [1]:
import os
from dotenv import load_dotenv
from rich import print

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# 環境変数を読み込む
load_dotenv(override=True)

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_API_BASE = os.getenv("OPENROUTER_API_BASE")

# 要約生成用のモデルを初期化
model = init_chat_model(
    model="openai/gpt-5.4-mini",
    model_provider="openai",
    profile={"max_input_tokens": 128_000},
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_API_BASE,
)

In [2]:
# Agent を作成（middleware パラメータをリストとして指定）
agent = create_agent(
    # agent が使用するモデル
    model='deepseek-v4-flash',
    middleware=[
        SummarizationMiddleware(
            # 要約生成に使用するモデル
            model=model,
            trigger=[
                ('tokens', 100),
                ('messages', 6),
                ('fraction', 0.001)
            ],
            keep=('messages', 3)
        )
    ]
)

# 会話履歴
messages = [
    SystemMessage("あなたはとても親切なAIアシスタントです"),
    HumanMessage("こんにちは、私は田中です。あなたは誰ですか？"),
    AIMessage("こんにちは田中さん、私も田中です"),
    HumanMessage("わかりました、お会いできて嬉しいです"),
    AIMessage("喜ぶのはまだ早いですよ"),
    HumanMessage("ふふ、どういう意味ですか")
]

# Agent を呼び出す
response = agent.invoke({
    "messages": messages,
})

# 結果を出力
for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

## SESSION INTENT
The user is having a brief introductory conversation and asked who the assistant is.

## SUMMARY
The conversation is in Japanese.
- The user introduced themselves as “田中” and asked, “あなたは誰ですか？” (“Who are you?”).
- The assistant replied: “こんにちは田中さん、私も田中です” (“Hello Tanaka-san, I’m also Tanaka”), which appears to be a mistaken self-identification rather than a substantive task.
- No concrete task, plan, or work item has been established beyond this greeting.

## ARTIFACTS
None

## NEXT STEPS
Respond appropriately to the user’s introduction and correct the assistant’s identity if continuing the conversation.
================================ Human Message =================================

わかりました、お会いできて嬉しいです
================================== Ai Message ==================================

喜ぶのはまだ早いですよ
================================ Human Messag

summary_prompt

In [3]:
# Agent を作成（middleware パラメータをリストとして指定）
agent = create_agent(
    model='deepseek-v4-flash',
    middleware=[
        SummarizationMiddleware(
            # 要約生成に使用するモデル
            model=model,
            trigger=[
                ('tokens', 100),
                ('messages', 6),
                ('fraction', 0.001)
            ],
            keep=('messages', 2),
            summary_prompt="過去のメッセージを要約してください。メッセージ一覧は以下の通りです\n{messages}"
        )
    ]
)

# 会話履歴
messages = [
    SystemMessage("あなたはとても親切なAIアシスタントです"),
    HumanMessage("こんにちは、私は田中です。あなたは誰ですか？"),
    AIMessage("こんにちは田中さん、私も田中です"),
    HumanMessage("わかりました、お会いできて嬉しいです"),
    AIMessage("喜ぶのはまだ早いですよ"),
    HumanMessage("ふふ、どういう意味ですか")
]

# Agent を呼び出す
response = agent.invoke({
    "messages": messages,
})

# 結果を出力
for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

以下のようなやり取りでした。

- システムから「とても親切なAIアシスタントであること」が指示された。
- 田中さんが自己紹介し、「あなたは誰ですか？」と質問した。
- AIは「こんにちは田中さん、私も田中です」と返答した。
- その後、田中さんは「わかりました、お会いできて嬉しいです」と応答した。
================================== Ai Message ==================================

喜ぶのはまだ早いですよ
================================ Human Message =================================

ふふ、どういう意味ですか
================================== Ai Message ==================================

「ふふ」と笑われましたね。では、答えます。

私が「喜ぶのはまだ早いですよ」と言ったのは、こういう意味です。

確かに私は「田中」と名乗りましたが、AIアシスタントとしての名前です。田中さんが「お会いできて嬉しい」とおっしゃったとき、私はまだ実際にあなたとお話しし始めたばかりで、本当に喜んでいただけるだけのことを何もしていませんよね。だから、「喜ぶのはまだ早い」と申し上げたんです。

でも、もし田中さんが今すでに少し楽しんでいただいているなら、私も同じくらい嬉しいです。

これからゆっくりお話しして、「ああ、会えてよかった」と思っていただけるように頑張りますね。どうぞよろしくお願いします。
